# 11 • Análises reproduzíveis e validação

## Preparar esta sessão
Execute as células abaixo antes de começar a aula. Elas localizam os arquivos e preparam os objetos necessários. Você não precisa decorar essa preparação agora.

In [1]:
from pathlib import Path
import pandas as pd
import math
BASE = Path.cwd()
if not (BASE / 'dados').exists() and (BASE.parent / 'dados').exists():
    BASE = BASE.parent
DADOS = BASE / 'dados'
SAIDAS = BASE / 'resultados'
SAIDAS.mkdir(exist_ok=True)
assert (DADOS/'penguins.csv').exists(), 'Descompacte o curso inteiro e abra na pasta correta.'


## Preparação independente
Execute esta célula para recriar os objetos necessários dos módulos anteriores.

In [2]:
p = pd.read_csv(DADOS / 'penguins.csv')
print(p.head(3).to_string(index=False))
print('Dimensões:', p.shape)
print('Colunas:', p.columns.tolist())
print(type(p['species']).__name__)

mascara = p['species'].eq('Adelie') & p['body_mass_g'].gt(4000)
print(p.loc[mascara, ['species', 'island', 'body_mass_g']].head())
print(p.iloc[:3, :2])

qualidade = pd.DataFrame({'ausentes':p.isna().sum(), 'percentual':p.isna().mean().mul(100)})
print(qualidade)
medidos = p.dropna(subset=['body_mass_g']).copy()
print('Observações com massa:', len(medidos))

enriquecido = p.assign(body_mass_kg=p['body_mass_g']/1000)
print(enriquecido[['body_mass_g','body_mass_kg']].head())
print(p['species'].astype('string').str.lower().head())

resumo = p.groupby('species', as_index=False, observed=True).agg(
    observacoes=('body_mass_g','size'),
    massas_validas=('body_mass_g','count'),
    massa_media_g=('body_mass_g','mean'))
print(resumo)

nomes = pd.DataFrame({'species':['Adelie','Chinstrap','Gentoo'],
                      'nome_pt':['Adélia','Barbicha','Papua']})
cruzado = p.merge(nomes, on='species', how='left', validate='many_to_one', indicator=True)
print(cruzado['_merge'].value_counts())

r = pd.read_csv(DADOS/'online_retail.csv', dtype={
    'InvoiceNo':'string','StockCode':'string','CustomerID':'string'})
r['InvoiceDate'] = pd.to_datetime(r['InvoiceDate'], errors='coerce')
cancelada = r['InvoiceNo'].str.upper().str.startswith('C', na=False)
elegivel = (r['Quantity'].gt(0) & r['UnitPrice'].gt(0)
            & r['InvoiceDate'].notna() & ~cancelada)
v = r.loc[elegivel].copy()
v['valor'] = v['Quantity'] * v['UnitPrice']
print('Linhas brutas:',len(r),'Linhas elegíveis:',len(v))
print(r.isna().sum())

mensal = v.set_index('InvoiceDate')['valor'].resample('MS').sum(min_count=1)
print(mensal)
print('Cobertura:',r['InvoiceDate'].min(),r['InvoiceDate'].max())

diario = v.groupby(v['InvoiceDate'].dt.normalize())['valor'].sum()
diario = diario.reindex(pd.date_range(diario.index.min(),diario.index.max(),freq='D'), fill_value=0)
print(diario.head())
# Hipótese: dias não registrados dentro da cobertura tiveram zero venda elegível.

identificados = v.dropna(subset=['CustomerID']).copy()
referencia = v['InvoiceDate'].max().normalize() + pd.Timedelta(days=1)
rfm = identificados.groupby('CustomerID').agg(
    ultima_compra=('InvoiceDate','max'), frequencia=('InvoiceNo','nunique'), monetario=('valor','sum'))
rfm['recencia_dias'] = (referencia-rfm['ultima_compra'].dt.normalize()).dt.days
print(rfm.head())

antes = p['species'].memory_usage(deep=True)
categorica = p['species'].astype('category')
print('Memória original:',antes,'bytes; categórica:',categorica.memory_usage(deep=True))

species    island  bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g    sex  year
 Adelie Torgersen            39.1           18.7              181.0       3750.0   male  2007
 Adelie Torgersen            39.5           17.4              186.0       3800.0 female  2007
 Adelie Torgersen            40.3           18.0              195.0       3250.0 female  2007
Dimensões: (344, 8)
Colunas: ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex', 'year']
Series
   species     island  body_mass_g
7   Adelie  Torgersen       4675.0
9   Adelie  Torgersen       4250.0
14  Adelie  Torgersen       4400.0
17  Adelie  Torgersen       4500.0
19  Adelie  Torgersen       4200.0
  species     island
0  Adelie  Torgersen
1  Adelie  Torgersen
2  Adelie  Torgersen
                   ausentes  percentual
species                   0    0.000000
island                    0    0.000000
bill_length_mm            2    0.581395
bill_depth_mm             

### Entendendo a função do exemplo

`def preparar_vendas(bruto):` define uma função chamada preparar_vendas que recebe uma tabela pelo parâmetro bruto. As linhas indentadas pertencem à função. Elas só são executadas quando você chama `preparar_vendas(r)`. return entrega a tabela preparada para quem chamou. O nome bruto é local à função e evita depender diretamente de uma variável global chamada r.

obrigatorias é um conjunto, criado com chaves; não é um dicionário, porque não contém pares chave:valor. A diferença entre conjuntos encontra os nomes que faltam no arquivo. `if faltantes:` entra no bloco quando há alguma falta. raise ValueError interrompe a transformação com uma mensagem clara. A expressão f'...' permite inserir o valor de uma variável dentro de um texto.

Um teste pequeno deve ter resposta conhecida independentemente do algoritmo. Crie manualmente quatro linhas: uma compra válida, um cancelamento, um preço zero e uma data inválida. Se só a primeira deve passar, confira o identificador e o valor esperado dessa linha. Isso verifica uma regra concreta; comparar duas cópias do mesmo cálculo pode reproduzir o mesmo erro nos dois lados. A função do exemplo verifica colunas e aplica regras, mas não é um contrato completo para qualquer entrada: antes de usá-la com outro sistema, acrescente validação e conversão explícita de Quantity e UnitPrice.

### Aprofundamento e cuidados

Um notebook profissional precisa funcionar com o ambiente recém-iniciado, na ordem documentada. Separe entrada, transformação e saída. Registre versões, fonte, data de obtenção e regras. Funções com entradas e retornos explícitos são mais fáceis de revisar do que alterações espalhadas em variáveis globais.

Um contrato define colunas obrigatórias, tipos e restrições. Valide antes de publicar. Os checks devem testar o significado: valor é quantidade vezes preço, chave tem a unicidade esperada, soma do relatório reconcilia com a base elegível. Testar apenas se uma função devolve um DataFrame é insuficiente.

Use `pd.testing` para comparações estruturais; para somas de ponto flutuante use tolerância. No código de produção prefira exceções explícitas para contratos, pois asserts podem ser desativados. Para exemplos de estudo e verificações locais, asserts deixam as condições visíveis. Não capture todas as exceções e continue silenciosamente. Registre linhas rejeitadas e motivos; não inclua dados de clientes em logs públicos.


## Exemplo guiado

In [3]:
def preparar_vendas(bruto):
    obrigatorias = {'InvoiceNo','InvoiceDate','Quantity','UnitPrice','CustomerID','Country'}
    faltantes = obrigatorias - set(bruto.columns)
    if faltantes:
        raise ValueError(f'Colunas ausentes: {sorted(faltantes)}')
    t = bruto.copy()
    t['InvoiceDate'] = pd.to_datetime(t['InvoiceDate'], errors='coerce')
    cancelamento = t['InvoiceNo'].astype('string').str.upper().str.startswith('C', na=False)
    ok = t['Quantity'].gt(0) & t['UnitPrice'].gt(0) & t['InvoiceDate'].notna() & ~cancelamento
    saida = t.loc[ok].copy()
    saida['valor'] = saida['Quantity']*saida['UnitPrice']
    return saida

v2 = preparar_vendas(r)
pd.testing.assert_frame_equal(v, v2)
print('Transformação reproduzida')

Transformação reproduzida


## Exercício 11.1
Confirme que preparar_vendas não modifica a tabela de entrada.

In [4]:
copia = r.copy(deep=True)
_ = preparar_vendas(r)
pd.testing.assert_frame_equal(r,copia)
print('Entrada preservada')

Entrada preservada


## Exercício 11.2
Demonstre a falha de contrato ao remover Country.

In [5]:
try:
    preparar_vendas(r.drop(columns='Country'))
except ValueError as erro:
    print(erro)

Colunas ausentes: ['Country']


## Exercício 11.3
Valide quantidade, preço, data e valor nas vendas elegíveis.

In [6]:
assert v['Quantity'].gt(0).all()
assert v['UnitPrice'].gt(0).all()
assert v['InvoiceDate'].notna().all()
pd.testing.assert_series_equal(v['valor'],(v['Quantity']*v['UnitPrice']).rename('valor'))
print('Regras conferidas')

Regras conferidas


## Exercício 11.4
Reconcilie a receita mensal com a receita total.

In [7]:
assert math.isclose(mensal.sum(),v['valor'].sum(),rel_tol=1e-10)
print('Resumo mensal reconciliado')

Resumo mensal reconciliado


## Exercício 11.5
Produza um registro com versão do pandas, linhas lidas, linhas elegíveis e intervalo observado.

In [8]:
registro = {'pandas':pd.__version__,'linhas_brutas':len(r),'linhas_elegiveis':len(v),'inicio':str(r['InvoiceDate'].min()),'fim':str(r['InvoiceDate'].max())}
print(registro)

{'pandas': '3.0.1', 'linhas_brutas': 541909, 'linhas_elegiveis': 530104, 'inicio': '2010-12-01 08:26:00', 'fim': '2011-12-09 12:50:00'}
